# Explore SOCIB WMOP Model — Full Western Mediterranean Domain
* Access data from the **SOCIB THREDDS Server using OPeNDAP** (same approach as `socib-wmop-best.ipynb`)
* Full WMOP domain: 5.8°W – 9.2°E, 34.9°N – 44.7°N
* Variables available: `temp`, `salt`, `u`, `v`, `ubar`, `vbar`, `zeta`
* Visualize and explore with **HoloViz** (interactive maps with time slider)

In [ ]:
import xarray as xr
import hvplot.xarray
import panel as pn
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# Rolling 5-day window ending now
stop_time  = pd.Timestamp.now()
start_time = stop_time - pd.DateOffset(days=5)

print(f"Start Time: {start_time}")
print(f"Stop Time:  {stop_time}")

## Open the WMOP "Best Time Series" via OPeNDAP

The SOCIB THREDDS server exposes the WMOP surface Best Time Series (most recent model run) at:
```
http://thredds.socib.es/thredds/dodsC/operational_models/oceanographical/hydrodynamics/
    model_run_aggregation/wmop_surface/wmop_surface_best.ncd
```
Opening with `xr.open_dataset()` is lazy — no data are downloaded until `.load()` or `.sel()` is called.

In [ ]:
dap_url = (
    'http://thredds.socib.es/thredds/dodsC/'
    'operational_models/oceanographical/hydrodynamics/'
    'model_run_aggregation/wmop_surface/wmop_surface_best.ncd'
)

ds = xr.open_dataset(dap_url)
ds

In [ ]:
# Inspect spatial extent and available variables
print('Variables:', list(ds.data_vars))
print('Lat range:', float(ds['lat_rho'].min()), '–', float(ds['lat_rho'].max()), '°N')
print('Lon range:', float(ds['lon_rho'].min()), '–', float(ds['lon_rho'].max()), '°E')
print('Time range:', str(ds['time'].values[0])[:16], '–', str(ds['time'].values[-1])[:16])

## Colour scale

In [ ]:
temp_clim = (13, 26)   # °C — full Western Med seasonal range
salt_clim = (36, 39)   # PSU
zeta_clim = (-0.3, 0.3)  # m — sea surface height anomaly

## Sea Surface Temperature — interactive map with time slider

In [ ]:
%%time
da_temp = ds['temp'].sel(time=slice(start_time, stop_time)).load()

temp_viz = da_temp.hvplot.quadmesh(
    x='lon_rho', y='lat_rho',
    rasterize=True,
    cmap='turbo',
    geo=True, tiles='OSM',
    clim=temp_clim,
    clabel='°C',
    title='WMOP Sea Surface Temperature (°C) — Full Western Mediterranean',
    frame_width=800,
    widgets={'time': pn.widgets.Select},
)

In [ ]:
temp_viz

## Sea Surface Salinity — interactive map with time slider

In [ ]:
%%time
da_salt = ds['salt'].sel(time=slice(start_time, stop_time)).load()

salt_viz = da_salt.hvplot.quadmesh(
    x='lon_rho', y='lat_rho',
    rasterize=True,
    cmap='viridis',
    geo=True, tiles='OSM',
    clim=salt_clim,
    clabel='PSU',
    title='WMOP Sea Surface Salinity (PSU) — Full Western Mediterranean',
    frame_width=800,
    widgets={'time': pn.widgets.Select},
)

In [ ]:
salt_viz

## Sea Surface Height (zeta) — interactive map with time slider

In [ ]:
%%time
da_zeta = ds['zeta'].sel(time=slice(start_time, stop_time)).load()

zeta_viz = da_zeta.hvplot.quadmesh(
    x='lon_rho', y='lat_rho',
    rasterize=True,
    cmap='RdBu_r',
    geo=True, tiles='OSM',
    clim=zeta_clim,
    clabel='m',
    title='WMOP Sea Surface Height anomaly (m) — Full Western Mediterranean',
    frame_width=800,
    widgets={'time': pn.widgets.Select},
)

In [ ]:
zeta_viz

## Surface Currents (u, v) — vector overlay at latest time step

In [ ]:
%%time
compare_time = stop_time

da_temp_snap = ds['temp'].sel(time=compare_time, method='nearest').load()
da_u_snap    = ds['u'].sel(time=compare_time, method='nearest').load()
da_v_snap    = ds['v'].sel(time=compare_time, method='nearest').load()

# Background: SST
sst_map = da_temp_snap.hvplot.quadmesh(
    x='lon_rho', y='lat_rho',
    rasterize=True,
    cmap='turbo',
    geo=True, tiles='OSM',
    clim=temp_clim,
    clabel='°C',
    title=f"WMOP SST + surface currents — {str(da_temp_snap.time.values)[:13]}",
    frame_width=800,
)

# Currents: subsample every 5th point to avoid overplotting
import numpy as np
step = 5
lon2d = ds['lon_rho'].values[::step, ::step]
lat2d = ds['lat_rho'].values[::step, ::step]
u2d   = da_u_snap.values[::step, ::step]
v2d   = da_v_snap.values[::step, ::step]

import pandas as pd
import hvplot.pandas

df_uv = pd.DataFrame({
    'longitude': lon2d.ravel(),
    'latitude':  lat2d.ravel(),
    'u': u2d.ravel(),
    'v': v2d.ravel(),
    'speed': np.hypot(u2d, v2d).ravel(),
}).dropna()

vec_viz = df_uv.hvplot.vectorfield(
    x='longitude', y='latitude',
    angle=np.arctan2(df_uv['v'], df_uv['u']),
    mag='speed',
    geo=True,
    color='speed',
    cmap='gray',
    scale=0.08,
    hover=False,
)

sst_map * vec_viz

## Basin-averaged daily SST time series — full WMOP domain

In [ ]:
ts_temp = da_temp.mean(dim=['eta_rho', 'xi_rho'])

ts_temp.hvplot.line(
    x='time',
    ylabel='Temperature (°C)',
    title='WMOP basin-averaged SST — Full Western Mediterranean',
    color='tomato',
    line_width=2,
    frame_width=700,
)